# Logistic regression hyperparameters

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.simplefilter("ignore")

X, y = load_breast_cancer(return_X_y=True)
print(X.shape, np.bincount(y))

## Penalty type (scikit-learn 1.8+: l1_ratio and C instead of penalty)

In [ ]:
settings = {
    "no penalty (C=inf)":           dict(C=np.inf),
    "L2, C=1 (default)":            dict(C=1.0),
    "L1, C=1":                      dict(C=1.0, l1_ratio=1, solver="saga", max_iter=20000),
    "Elastic Net 0.5, C=1":         dict(C=1.0, l1_ratio=0.5, solver="saga", max_iter=20000),
}
for name, kw in settings.items():
    m = make_pipeline(StandardScaler(), LogisticRegression(**kw))
    cv = cross_val_score(m, X, y, cv=5).mean()
    nz = (np.abs(m.fit(X, y)[-1].coef_) > 1e-8).sum()
    print(f"{name:28s} CV accuracy {cv:.3f}, non-zero coefficients {nz}")

## C: smaller means stronger regularisation

In [ ]:
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C))
    print(f"C = {C:>7}: CV accuracy {cross_val_score(m, X, y, cv=5).mean():.3f}, largest |coef| {np.abs(m.fit(X, y)[-1].coef_).max():.3f}")

## class_weight on imbalanced data

In [ ]:
Xi, yi = make_classification(n_samples=2000, n_features=5, weights=[0.95], random_state=0, flip_y=0.01)
a, b, c, d = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
print("test class counts", np.bincount(d))
for cw in [None, "balanced"]:
    p = LogisticRegression(class_weight=cw).fit(a, c).predict(b)
    print(f"class_weight={cw}: recall {recall_score(d, p):.3f}, precision {precision_score(d, p):.3f}")

## Solver and penalty must match

In [ ]:
try:
    LogisticRegression(l1_ratio=1, solver="lbfgs").fit(X, y)
except ValueError as e:
    print("ValueError:", e)